# 02 — Training and validation
Run a small executable training example, inspect the frozen configuration, and optionally run the real repeated-CV workflow. The synthetic example demonstrates the API; its score is not competition evidence.

In [ ]:
import os
from pathlib import Path
for name in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[name] = '1'
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits
from copilote_b.config import load_config
from copilote_b.data import parse_line
from copilote_b.features import prepare
from copilote_b.model import CropSVM
from copilote_b.ensemble import blend, metrics
from copilote_b.pipeline import initialize, load_training
from copilote_b.evaluation import cross_validate, evaluate_blend
PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/copilote_b').is_dir())
default_data = PROJECT.parent / 'data' if (PROJECT.parent / 'data/train.csv').exists() else PROJECT / 'data'
DATA = Path(os.environ.get('COPILOTE_DATA', default_data)).resolve()
OUTPUT = Path(os.environ.get('COPILOTE_OUTPUT', PROJECT / 'outputs/reproduction')).resolve()
config = load_config(PROJECT / 'configs/solution.json')
display(pd.DataFrame(config['members']).T)
display(pd.Series(config['weights'], name='submitted_weight'))

## Small end-to-end example
Both users use Firefox so browser conditioning cannot solve this example by itself. Every member learns its own vocabulary and crop-weighted SVM.

In [ ]:
lines = ['Firefox,Open(Account)<A>$C$,Edit1,Save,t5'] * 3 + ['Firefox,Open(Planning)<P>$H$,Schedule,Validate,t10'] * 3
traces = [prepare(parse_line(line, i + 1)) for i, line in enumerate(lines)]
y = np.array([0, 0, 0, 1, 1, 1])
validation = [prepare(parse_line(lines[0], 1)), prepare(parse_line(lines[-1], 2))]
predictions = {}
with threadpool_limits(limits=1):
    for name, parameters in config['members'].items():
        model = CropSVM(parameters, config['seed']).fit(traces, y)
        predictions[name] = model.predict_proba(validation)
toy_blend = blend(predictions, config['weights'])
display(pd.DataFrame(toy_blend, columns=['user_0', 'user_1']))
print('Synthetic demonstration:', metrics(np.array([0, 1]), toy_blend))
assert toy_blend.argmax(axis=1).tolist() == [0, 1]

## Real-data CV (opt-in)
The full run uses five folds and seeds 137/271. It refits preprocessing and browser support inside each fold, and saves OOF scores for every member. For a quick real first-fold check, set `RUN_REAL_CV=True`, `REPEATS=1`, `FOLDS=1`. Partial runs are never described as complete repeated CV.

In [ ]:
RUN_REAL_CV = False
REPEATS, FOLDS = 2, 5
if RUN_REAL_CV:
    initialize(DATA, OUTPUT, config)
    _, traces, y, classes = load_training(DATA)
    display(cross_validate(traces, y, classes, config, OUTPUT, REPEATS, FOLDS))
elif (OUTPUT / 'cv/results.csv').exists():
    display(pd.read_csv(OUTPUT / 'cv/results.csv'))
else:
    print('Run the CLI cv stage, or opt in above, to generate real-data results.')

## OOF weight fitting is an apparent-score diagnostic
The following is opt-in and requires all 10 folds. Newly optimized weights are saved separately; the frozen submitted weights are unchanged. For independent weight evaluation, run `python run.py nested --data ../data`. Candidate identities were selected during earlier exploratory CV, so some selection optimism remains.

In [ ]:
RUN_WEIGHT_SEARCH = False
if RUN_WEIGHT_SEARCH:
    initialize(DATA, OUTPUT, config)
    _, _, y, classes = load_training(DATA)
    display(evaluate_blend(y, classes, config, OUTPUT))
else:
    print('Weight search disabled; prediction uses configs/solution.json.')

## Next
Train the final models with `python run.py train --data ../data`, then open [submission and verification](03_submission_and_verification.ipynb). See [validation notes](../docs/VALIDATION.md) for historical metrics and their limitations.